<a href="https://colab.research.google.com/github/janithcyapa/Statistical-Learning-e20452/blob/main/Assignments/Assignment_05_Gaussian_Process_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment : Gaussian Process Regression & Linear Regression**
### ME526 : Introduction to Statistical Learning
---
> YAPA W.S.P.Y.J.C.
>
> E/20/452
---

## **Gaussian Process Regression**

Modeling both the Heating Load (Y1) and Cooling Load (Y2) as a single parameter Gaussian Process need to handle two dependent thermal loads simultaneously, we therefore General Multivariate Formulation is used here.

Instead of two independent scalar processes, define  observation $Y_g$ at any building configuration g as a vector in $\mathbb{R}^2$,
$$Y_g =X_g + ν_g$$

where $X_g ∈ R_2$ is the latent clean thermal load, and $ν_g ∼ N(0,Σν​)$ is the noise vector.
\
\
If we have $n$ building observations, our stacked observation vector $\mathscr{Y}_n$ is in $\mathbb{R}^{2n}$. The joint marginal distribution becomes:


$$\mathscr{Y}_n \sim \mathscr{N} \left( \mu_{Y,n}, K_n + I_n \otimes \Sigma_\nu \right)$$

Here, the covariance matrix $K_n$ is built from a matrix-valued kernel $\kappa(g, g') \in \mathbb{R}^{2 \times 2}$. The diagonal blocks represent the variance of heating and cooling loads independently, while the off-diagonal elements capture the cross-correlation between them.

To estimate the thermal loads $X_g$ for a new building configuration, the predictive mean relies on the block matrix inversion,

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \mu_g + K_{g,n} \left( K_n + I_n \otimes \Sigma_\nu \right)^{-1} \left( y_n - \mu_{Y,n} \right)$$

By modeling them together, the GP leverages the shared architectural features (X1 to X8) and the intrinsic thermodynamic coupling between heating and cooling to create a more robust predictive distribution.

In [1]:
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

# --- 1. Data Retrieval ---
kagglepath = "elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)
df = pd.read_csv(os.path.join(path, "ENB2012_data.csv"))

# The dataset features X1: Relative Compactness to X8: Glazing Area Distribution
# Responses Y1: Heating Load, Y2: Cooling Load
feature_cols = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']
target_cols = ['Y1', 'Y2']

X = df[feature_cols].values
Y = df[target_cols].values

# --- 2. Exploratory Data Analysis using Custom Function ---
from data_analysis import DataInspector
inspector = DataInspector()
inspector.df = df

# Explore the conditional distributions before fitting
print("Computing conditional normals for building parameters...")
eda_result = inspector.compute_and_plot_conditional_normal(
    x_columns=feature_cols,
    y_columns=target_cols
)
display(eda_result)

# --- 3. Data Preprocessing ---
# GPs are highly sensitive to distance metrics, so feature scaling is mandatory
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# --- 4. Gaussian Process Modeling ---
# We use a Matern kernel to handle potential 'jaggedness' in non-linear building data,
# combined with a WhiteKernel to account for simulation/measurement noise.
kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(8), nu=1.5) + WhiteKernel(noise_level=1, noise_level_bounds=(1e-5, 1e1))

# By passing a 2D array for Y, sklearn automatically handles multi-target regression.
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True, random_state=42)

print("Fitting the Multi-Output Gaussian Process...")
gp.fit(X_train_scaled, Y_train)

# --- 5. Prediction and Evaluation ---
Y_pred, Y_std = gp.predict(X_test_scaled, return_std=True)

# Calculate R^2 for both targets
r2_heating = gp.score(X_test_scaled, Y_test[:, 0].reshape(-1, 1)) # Wait, score takes joint if y is 2D, let's calculate manually or use joint score
r2_joint = gp.score(X_test_scaled, Y_test)
print(f"Optimized Kernel: {gp.kernel_}")
print(f"Joint R^2 Score on Test Set: {r2_joint:.4f}")

# Plotting the results for Heating Load (Y1) as an example
plt.figure(figsize=(10, 5))
plt.errorbar(range(40), Y_test[:40, 0], yerr=1.96 * Y_std[:40], fmt='o', color='black',
             ecolor='lightgray', elinewidth=3, capsize=0, label='Actual Heating Load (Y1)')
plt.plot(range(40), Y_pred[:40, 0], 'rx', label='GP Predictive Mean')
plt.title("GPR Predictions vs Actual: Heating Load (First 40 Test Samples)")
plt.xlabel("Sample Index")
plt.ylabel("Heating Load (kWh/m²)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.


ModuleNotFoundError: No module named 'data_analysis'